# 04 — Model Training

**Objective:** Train and tune four supervised classification models for predicting high-risk pregnancies.

## Models
1. Logistic Regression (interpretable baseline)
2. Decision Tree Classifier (rule-based)
3. Random Forest Classifier (ensemble)
4. Deep Neural Network / MLP (deep learning)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.preprocess import load_dataset, handle_imbalance
from src.models import logistic_regression, decision_tree, random_forest
from src.models import lstm_model as mlp_module

%matplotlib inline

In [ ]:
# Load processed data
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

X_train = train_df.drop(columns=['risk_label'])
y_train = train_df['risk_label']
X_val = val_df.drop(columns=['risk_label'])
y_val = val_df['risk_label']
X_test = test_df.drop(columns=['risk_label'])
y_test = test_df['risk_label']

feature_names = X_train.columns.tolist()

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
print(f'Train risk prevalence: {y_train.mean():.3f}')
print(f'Features ({len(feature_names)}): {feature_names}')

# Handle class imbalance with SMOTE for sklearn models
from src.data.preprocess import handle_imbalance

X_train_smote, y_train_smote = handle_imbalance(X_train, y_train, method='smote')
class_weights = handle_imbalance(X_train, y_train, method='weights')

In [ ]:
## 1. Logistic Regression
print("=" * 50)
print("Training: Logistic Regression")
print("=" * 50)

lr_model, lr_params = logistic_regression.tune_hyperparameters(X_train_smote, y_train_smote, cv=5)

# Validation performance
from src.evaluation.metrics import compute_metrics
y_val_pred = lr_model.predict(X_val)
y_val_prob = lr_model.predict_proba(X_val)[:, 1]
lr_val = compute_metrics(y_val, y_val_pred, y_val_prob)
print(f"\nValidation — F1: {lr_val['f1_score']:.4f}, AUC: {lr_val['roc_auc']:.4f}")

# Feature importance
lr_imp = logistic_regression.get_feature_importance(lr_model, feature_names)
print(f"\nTop 5 features by |coefficient|:")
print(lr_imp.head().to_string(index=False))

logistic_regression.save_model(lr_model, '../models/logistic_regression.joblib')

## 2. Decision Tree Classifier
print("=" * 50)
print("Training: Decision Tree")
print("=" * 50)

dt_model, dt_params = decision_tree.tune_hyperparameters(X_train_smote, y_train_smote, cv=5)

y_val_pred = dt_model.predict(X_val)
y_val_prob = dt_model.predict_proba(X_val)[:, 1]
dt_val = compute_metrics(y_val, y_val_pred, y_val_prob)
print(f"\nValidation — F1: {dt_val['f1_score']:.4f}, AUC: {dt_val['roc_auc']:.4f}")

dt_imp = decision_tree.get_feature_importance(dt_model, feature_names)
print(f"\nTop 5 features:")
print(dt_imp.head().to_string(index=False))

# Print human-readable rules
rules = decision_tree.get_tree_rules(dt_model, feature_names, max_depth=3)
print(f"\nDecision Rules (depth ≤ 3):\n{rules}")

decision_tree.save_model(dt_model, '../models/decision_tree.joblib')

In [ ]:
## 3. Random Forest Classifier
print("=" * 50)
print("Training: Random Forest")
print("=" * 50)

rf_model, rf_params = random_forest.tune_hyperparameters(X_train_smote, y_train_smote, cv=5, n_iter=30)

y_val_pred = rf_model.predict(X_val)
y_val_prob = rf_model.predict_proba(X_val)[:, 1]
rf_val = compute_metrics(y_val, y_val_pred, y_val_prob)
print(f"\nValidation — F1: {rf_val['f1_score']:.4f}, AUC: {rf_val['roc_auc']:.4f}")

rf_imp = random_forest.get_feature_importance(rf_model, feature_names)
print(f"\nTop 5 features:")
print(rf_imp.head().to_string(index=False))

random_forest.save_model(rf_model, '../models/random_forest.joblib')

## 4. Deep Neural Network (MLP)
print("=" * 50)
print("Training: Deep Neural Network (MLP)")
print("=" * 50)

X_train_np = mlp_module.prepare_input(X_train)
X_val_np = mlp_module.prepare_input(X_val)
n_features = X_train_np.shape[1]

model = mlp_module.build_model(n_features, learning_rate=0.001)
model.summary()

# Train with class weights (better than SMOTE for neural nets)
history = mlp_module.train_model(
    model, X_train_np, y_train.values,
    X_val_np, y_val.values,
    epochs=100, batch_size=32, class_weight=class_weights,
)

In [ ]:
# MLP Training History
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Val Loss', linewidth=2)
axes[0].set_title('MLP Training Loss', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].set_title('MLP Training Accuracy', fontsize=12)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/mlp_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

mlp_module.save_model(model, '../models/mlp_model.keras')

## Training Summary

print("=" * 60)
print("MODEL TRAINING SUMMARY")
print("=" * 60)

summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'MLP'],
    'Best Params': [str(lr_params), str(dt_params), str(rf_params), 'Dense(64→32→16→1)'],
    'Val F1': [lr_val['f1_score'], dt_val['f1_score'], rf_val['f1_score'], None],
    'Val AUC': [lr_val['roc_auc'], dt_val['roc_auc'], rf_val['roc_auc'], None],
})

# Get MLP val metrics
X_val_np = mlp_module.prepare_input(X_val)
y_val_pred_mlp, y_val_prob_mlp = mlp_module.predict(model, X_val_np)
mlp_val = compute_metrics(y_val, y_val_pred_mlp, y_val_prob_mlp)
summary.loc[3, 'Val F1'] = mlp_val['f1_score']
summary.loc[3, 'Val AUC'] = mlp_val['roc_auc']

print(summary.to_string(index=False))
print("\nAll models saved to ../models/")

In [ ]:
# Placeholder — old stub cells replaced by training summary above

In [ ]:
# Placeholder — old stub cell replaced